# Claims Damage Ratio Prediction with CatBoost

In [31]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import optuna
from catboost import CatBoostRegressor, Pool


## Data and Config

In [2]:
# Load data
df = pd.read_csv("claims_cleaned.csv")
categorical_features = [
    'ratedFloodZone', 'occupancyType', 'basement', 
    'countyCode', 'elevatedBuildingIndicator'
]

In [3]:
# config settings
TARGET = "damageRatio"
CATEGORICAL = categorical_features
TEST_SIZE = 0.2
VAL_SIZE = 0.2
RANDOM_STATE = 30
N_TRIALS_MEAN = 30
N_TRIALS_VAR = 20
EARLY_STOPPING_ROUNDS = 200
CLIP_EPS = 1e-6

### Data prep

In [4]:
def train_val_test_split(df: pd.DataFrame):
    train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_df, val_df = train_test_split(train_df, test_size=VAL_SIZE, random_state=RANDOM_STATE)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

def build_features(df: pd.DataFrame):
    y = df[TARGET]
    X = df.drop(columns=[TARGET])
    for c in CATEGORICAL:
        if c in X.columns:
            X[c] = X[c].astype("category").astype(str)
    return X, y

def pool(X, y):
    cat_idx = [X.columns.get_loc(c) for c in CATEGORICAL if c in X.columns]
    return Pool(X, label=y, cat_features=cat_idx)

## Hyperparamter Tuning

In [15]:
def tune_catboost(X_train, y_train, X_val, y_val, n_trials, study_name):
    def objective(trial):
        params = dict(
            loss_function = 'RMSE',
            eval_metric = 'RMSE',
            depth = trial.suggest_int('depth', 4, 10),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
            bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 5.0),
            border_count=trial.suggest_int("border_count", 32, 255),
            min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 1, 200),
            random_seed=RANDOM_STATE,
            verbose=False,
            od_type="Iter"
        )

        model = CatBoostRegressor(**params)
        model.fit(
            pool(X_train, y_train),
            eval_set=pool(X_val, y_val),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            use_best_model=True
        )
        preds = model.predict(pool(X_val, None))
        rmse  = root_mean_squared_error(y_val, preds)
        return rmse
    
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=n_trials)
    return study.best_params


## Training Pipeline

In [6]:
def fit_gaussian(df: pd.DataFrame):
    train_df, val_df, test_df = train_val_test_split(df)
    X_tr, y_tr = build_features(train_df)
    X_val, y_val = build_features(val_df)

    # Stage 1: mean model
    best_mean = tune_catboost(X_tr, y_tr, X_val, y_val, N_TRIALS_MEAN, "mean_model")
    mean_model = CatBoostRegressor(**best_mean)
    mean_model.fit(pool(X_tr, y_tr), eval_set=pool(X_val, y_val),
    early_stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False)

    # Residuals for variance model
    mu_tr = mean_model.predict(pool(X_tr, None))
    mu_val = mean_model.predict(pool(X_val, None))
    var_trg_tr = np.log((y_tr.values - mu_tr) ** 2 + CLIP_EPS)
    var_trg_val = np.log((y_val.values - mu_val) ** 2 + CLIP_EPS)

    # Stage 2: variance model
    best_var = tune_catboost(X_tr, var_trg_tr, X_val, var_trg_val, N_TRIALS_VAR, "var_model")
    var_model = CatBoostRegressor(**best_var)
    var_model.fit(pool(X_tr, var_trg_tr), eval_set=pool(X_val, var_trg_val),
    early_stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False)

    return mean_model, var_model, test_df

## Evaluation

In [ ]:
def evaluate(mean_model, var_model, test_df):
    X_te, y_te = build_features(test_df)
    mu = mean_model.predict(pool(X_te, None))
    log_var = var_model.predict(pool(X_te, None))
    var = np.exp(log_var)


    rmse = root_mean_squared_error(y_te, mu)
    mae = mean_absolute_error(y_te, mu)
    r2 = r2_score(y_te, mu)
    nll = 0.5 * (np.log(2 * np.pi * var) + (y_te.values - mu) ** 2 / var)


    return {
    "RMSE": float(rmse),
    "MAE": float(mae),
    "R2": float(r2),
    "GaussianNLL": float(np.mean(nll)),
    }

In [16]:
mean_model, var_model, test_df = fit_gaussian(df)

[I 2025-09-08 22:56:07,605] A new study created in memory with name: mean_model
[I 2025-09-08 22:56:16,777] Trial 0 finished with value: 0.20869117101578602 and parameters: {'depth': 8, 'learning_rate': 0.19014825684697684, 'l2_leaf_reg': 4.357114478722029, 'bagging_temperature': 3.409679631002463, 'border_count': 180, 'min_data_in_leaf': 135}. Best is trial 0 with value: 0.20869117101578602.
[I 2025-09-08 22:56:37,634] Trial 1 finished with value: 0.2088192877407349 and parameters: {'depth': 10, 'learning_rate': 0.08906738264562877, 'l2_leaf_reg': 12.141624486601634, 'bagging_temperature': 4.586988884225799, 'border_count': 165, 'min_data_in_leaf': 32}. Best is trial 0 with value: 0.20869117101578602.
[I 2025-09-08 22:56:45,421] Trial 2 finished with value: 0.21003183169652911 and parameters: {'depth': 5, 'learning_rate': 0.08221638298625024, 'l2_leaf_reg': 2.048160660753981, 'bagging_temperature': 4.146209369723485, 'border_count': 70, 'min_data_in_leaf': 121}. Best is trial 0 with v

In [29]:
mu, y_te, var, nll, metrics =evaluate(mean_model, var_model, test_df)